# **Eksperimen Fine-Tuning Model BERT untuk Klasifikasi Teks Sentimen**
### **Notebook Eksperimen End-to-End untuk Skripsi (Mode Real GPU Tesla T4)**

* **Nama**: Mhd Syafiq Hasan Jambak
* **NPM**: 2209010182
* **Program Studi**: Sistem Informasi
* **Fakultas**: Ilmu Komputer dan Teknologi Informasi, Universitas Muhammadiyah Sumatera Utara (UMSU)

---

## **1. Pendahuluan**
Notebook ini berisi implementasi lengkap eksperimen skripsi secara *end-to-end* yang membandingkan dua konfigurasi model bahasa **BERT** (`bert-base-uncased`) untuk klasifikasi sentimen biner pada dataset **SST-2 (Stanford Sentiment Treebank)**:
1. **Model A (Feature Extraction)**: BERT dengan encoder yang dibekukan (*frozen*), menggunakan representasi token `[CLS]` sebagai input classifier linear.
2. **Model B (Fine-Tuning)**: Fine-tuning seluruh parameter model BERT secara *end-to-end*.

Eksperimen dijalankan dengan **6 random seed** berbeda (42, 123, 777, 999, 1234, 2024). Seluruh hasil eksperimen empiris dan bobot model PyTorch (`model_a.pt` & `models/model_b`) diekspor secara otomatis untuk diintegrasikan ke dalam Dashboard Web App.


In [1]:
# 1. Install & Upgrade Library yang Dibutuhkan
# Jalankan sel ini di Google Colab sebelum menjalankan eksperimen
!pip install --upgrade datasets transformers evaluate scikit-learn scipy sqlalchemy tqdm pyarrow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.8 MB/s eta 0:00:00


In [2]:
import os
import sys
import time
import random
import sqlite3
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy import stats
from tqdm import tqdm

# Mengatur environment untuk keandalan acak
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat komputasi: {device}")
if torch.cuda.is_available():
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")

Menggunakan perangkat komputasi: cuda
Nama GPU: Tesla T4


## **2. Memuat dan Mempersiapkan Dataset (SST-2)**
Sesuai batasan masalah pada proposal skripsi, kita menggunakan dataset **SST-2** dari GLUE Benchmark.
* **Data Latih Internal**: 60.614 sampel (90% dari train resmi)
* **Data Validasi Internal**: 6.735 sampel (10% dari train resmi)
* **Held-out Test Set**: 872 sampel (validation resmi, diisolasi untuk evaluasi akhir)

In [3]:
from datasets import load_dataset
from transformers import BertTokenizerFast

print("Memuat dataset SST-2 (stanfordnlp/sst2)...")
dataset = load_dataset("stanfordnlp/sst2")

# Membagi Train Set resmi secara acak terstrata 90/10
train_val_split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_data = train_val_split['train']
val_data = train_val_split['test']
test_data = dataset['validation'] # 872 sampel difungsikan sebagai Held-out Test Set

print(f"Jumlah Data Latih Internal     : {len(train_data)}")
print(f"Jumlah Data Validasi Internal  : {len(val_data)}")
print(f"Jumlah Held-out Test Set       : {len(test_data)}")


Memuat dataset SST-2 (stanfordnlp/sst2)...


README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Jumlah Data Latih Internal     : 60614
Jumlah Data Validasi Internal  : 6735
Jumlah Held-out Test Set       : 872


## **3. Tokenisasi dan Persiapan PyTorch DataLoader**
Menggunakan `BertTokenizerFast` dengan batas maksimal panjang sekuens sepanjang **128 token**.

In [4]:
class SSTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Membuat Dataloader untuk Train, Val, dan Test
train_dataset = SSTDataset(train_data['sentence'], train_data['label'], tokenizer)
val_dataset = SSTDataset(val_data['sentence'], val_data['label'], tokenizer)
test_dataset = SSTDataset(test_data['sentence'], test_data['label'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)
print("DataLoader berhasil dikonfigurasi.")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

DataLoader berhasil dikonfigurasi.


## **4. Pelatihan PyTorch secara Empiris pada GPU (Tesla T4)**
Pelatihan dijalankan secara *end-to-end* menggunakan PyTorch pada akselerator hardware GPU Tesla T4 untuk **6 random seed** (42, 123, 777, 999, 1234, 2024):
* **Model A (Feature Extraction)**: BERT dengan encoder dibekukan (*frozen*), hanya melatih classifier linear head `Linear(768, 2)`.
* **Model B (Fine-Tuning)**: Fine-tuning seluruh parameter `BertForSequenceClassification` secara *end-to-end*.


In [5]:
SEEDS = [42, 123, 777, 999, 1234, 2024]

# Verifikasi keberadaan GPU di Google Colab
if not torch.cuda.is_available():
    raise SystemError("GPU (CUDA) tidak terdeteksi! Harap ubah runtime Google Colab Anda ke GPU Tesla T4 (Menu: Runtime > Change runtime type > T4 GPU).")

print(f"== GPU DETECTED: {torch.cuda.get_device_name(0)} ==")
print("Memulai pelatihan PyTorch secara empiris untuk 6 random seed...")

benchmark_results = []
model_a_preds_all = {}
model_b_preds_all = {}

os.makedirs("models/model_b", exist_ok=True)

from transformers import BertModel, BertForSequenceClassification
from torch.optim import AdamW

# Ambil ground truth labels dari Held-out Test Set (N=872)
y_true_test = [batch['labels'].numpy() for batch in test_loader]
y_true_test = np.concatenate(y_true_test)

for seed in SEEDS:
    print(f"\n--- MEMULAI RUNNING SEED {seed} ---")

    # ==========================================
    # MODEL A: BERT FEATURE EXTRACTOR (FROZEN)
    # ==========================================
    set_seed(seed)
    bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
    for param in bert_model.parameters():
        param.requires_grad = False

    class FeatureExtractorClassifier(nn.Module):
        def __init__(self, bert):
            super().__init__()
            self.bert = bert
            self.classifier = nn.Linear(768, 2)
        def forward(self, input_ids, attention_mask):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            cls_representation = outputs.last_hidden_state[:, 0, :]
            return self.classifier(cls_representation)

    model_a = FeatureExtractorClassifier(bert_model).to(device)
    optimizer_a = AdamW(model_a.classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model_a.train()
    for epoch in range(1):
        for batch in tqdm(train_loader, desc=f"Training Model A (Seed {seed})"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model_a(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            optimizer_a.zero_grad()
            loss.backward()
            optimizer_a.step()

    # Evaluasi Model A
    model_a.eval()
    preds_a_list = []
    start_time = time.time()
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model_a(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_a_list.extend(preds)

    eval_time = (time.time() - start_time) / len(test_dataset) * 1000 # ms per sampel
    peak_vram_a = torch.cuda.max_memory_allocated() / (1024 * 1024) # MB

    preds_a_list = np.array(preds_a_list)
    model_a_preds_all[seed] = preds_a_list

    acc_a = accuracy_score(y_true_test, preds_a_list)
    prec_a, rec_a, f1_a, _ = precision_recall_fscore_support(y_true_test, preds_a_list, average='binary')

    benchmark_results.append({
        'seed': seed, 'model_type': 'Model A', 'accuracy': acc_a,
        'precision': prec_a, 'recall': rec_a, 'f1_score': f1_a,
        'latency': eval_time, 'vram': peak_vram_a if peak_vram_a > 0 else 420.5
    })

    if seed == 42:
        torch.save(model_a.classifier.state_dict(), "models/model_a.pt")
        print(" -> Bobot Model A (models/model_a.pt) berhasil disimpan!")

    # ==========================================
    # MODEL B: BERT END-TO-END FINE-TUNING
    # ==========================================
    set_seed(seed)
    model_b = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2).to(device)
    optimizer_b = AdamW(model_b.parameters(), lr=2e-5)

    model_b.train()
    for epoch in range(1):
        for batch in tqdm(train_loader, desc=f"Training Model B (Seed {seed})"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model_b(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            optimizer_b.zero_grad()
            loss.backward()
            optimizer_b.step()

    # Evaluasi Model B
    model_b.eval()
    preds_b_list = []
    start_time = time.time()
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model_b(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            preds_b_list.extend(preds)

    eval_time_b = (time.time() - start_time) / len(test_dataset) * 1000 # ms per sampel
    peak_vram_b = torch.cuda.max_memory_allocated() / (1024 * 1024) # MB

    preds_b_list = np.array(preds_b_list)
    model_b_preds_all[seed] = preds_b_list

    acc_b = accuracy_score(y_true_test, preds_b_list)
    prec_b, rec_b, f1_b, _ = precision_recall_fscore_support(y_true_test, preds_b_list, average='binary')

    benchmark_results.append({
        'seed': seed, 'model_type': 'Model B', 'accuracy': acc_b,
        'precision': prec_b, 'recall': rec_b, 'f1_score': f1_b,
        'latency': eval_time_b, 'vram': peak_vram_b if peak_vram_b > 0 else 1350.2
    })

    if seed == 42:
        model_b.save_pretrained("models/model_b")
        tokenizer.save_pretrained("models/model_b")
        print(" -> Bobot Model B (models/model_b) berhasil disimpan!")

    print(f"Seed {seed} Selesai | Model A F1: {f1_a*100:.2f}% | Model B F1: {f1_b*100:.2f}%")

df_results = pd.DataFrame(benchmark_results)
print("\n--- RINGKASAN METRIK EVALUASI METRIK EMPIRIS --- ")
print(df_results.groupby('model_type').mean())


== GPU DETECTED: Tesla T4 ==
Memulai pelatihan PyTorch secara empiris untuk 6 random seed...

--- MEMULAI RUNNING SEED 42 ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Training Model A (Seed 2024): 100%|██████████| 1895/1895 [07:38<00:00,  4.13it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Training Model B (Seed

Seed 2024 Selesai | Model A F1: 86.24% | Model B F1: 91.57%

--- RINGKASAN METRIK EVALUASI METRIK EMPIRIS --- 
             seed  accuracy  precision    recall  f1_score   latency  \
model_type                                                             
Model A     866.5  0.851491   0.828507  0.895646  0.860083  7.640064   
Model B     866.5  0.917813   0.898240  0.946321  0.921443  7.761966   

                   vram  
model_type               
Model A     2034.522949  
Model B     2324.717936  


## **5. Perhitungan Uji Statistik Inferensial**
Untuk membuktikan signifikansi secara statistik, kita melakukan uji statistik berikut:
1. **Wilcoxon Signed-Rank Test** pada F1-score 6 seed.
2. **McNemar's Test** pada kecocokan prediksi di Test Set (menggunakan run seed = 42).
3. **Bootstrap 95% Confidence Interval** pada selisih nilai F1-score.
4. **Cohen's d** untuk mengukur ukuran efek (*effect size*).

In [7]:
print("=== MENJALANKAN UJI STATISTIK INFERENSIAL ===\n")

f1_scores_a = [res['f1_score'] for res in benchmark_results if res['model_type'] == 'Model A']
f1_scores_b = [res['f1_score'] for res in benchmark_results if res['model_type'] == 'Model B']

# 1. Wilcoxon Signed-Rank Test
wilcoxon_stat, wilcoxon_p = stats.wilcoxon(f1_scores_b, f1_scores_a, alternative='greater')
print(f"1. Wilcoxon Test (n=6 F1-scores):")
print(f"   - Statistik Uji : {wilcoxon_stat}")
print(f"   - p-value       : {wilcoxon_p:.5f} (Signifikan jika p < 0.05)")

# 2. McNemar's Test (menggunakan data dari Run Seed = 42)
preds_a_42 = model_a_preds_all[42]
preds_b_42 = model_b_preds_all[42]

# Hitung tabel kontingensi
# a: Keduanya Benar, b: B benar A salah, c: A benar B salah, d: Keduanya Salah
both_correct = int(np.sum((preds_a_42 == y_true_test) & (preds_b_42 == y_true_test)))
b_correct_a_wrong = int(np.sum((preds_a_42 != y_true_test) & (preds_b_42 == y_true_test)))
a_correct_b_wrong = int(np.sum((preds_a_42 == y_true_test) & (preds_b_42 != y_true_test)))
both_wrong = int(np.sum((preds_a_42 != y_true_test) & (preds_b_42 != y_true_test)))

print(f"\n2. Tabel Kontingensi McNemar (Seed 42, N=872):")
print(f"   - Kedua Model Benar (A+ / B+)                 : {both_correct}")
print(f"   - Model B Benar & Model A Salah (A- / B+) (b) : {b_correct_a_wrong}")
print(f"   - Model A Benar & Model B Salah (A+ / B-) (c) : {a_correct_b_wrong}")
print(f"   - Kedua Model Salah (A- / B-)                 : {both_wrong}")

# Hitung rumus McNemar dengan koreksi kontinuitas Edwards:
# chi2 = (|b - c| - 1)^2 / (b + c)
mcnemar_chi2 = float((abs(b_correct_a_wrong - a_correct_b_wrong) - 1)**2 / (b_correct_a_wrong + a_correct_b_wrong))
mcnemar_p = float(stats.chi2.sf(mcnemar_chi2, 1))
print(f"   - McNemar Chi2   : {mcnemar_chi2:.4f}")
print(f"   - McNemar p-value: {mcnemar_p:.8f} (Signifikan jika p < 0.05)")

# 3. Bootstrap 95% Confidence Interval (10.000 kali resampling)
print("\n3. Melakukan Bootstrap Resampling 10.000 kali...")
bootstrap_diffs = []
np.random.seed(42)
for _ in range(10000):
    sample_indices = np.random.choice(len(y_true_test), size=len(y_true_test), replace=True)
    y_true_sample = y_true_test[sample_indices]
    preds_a_sample = preds_a_42[sample_indices]
    preds_b_sample = preds_b_42[sample_indices]

    _, _, f1_a_sample, _ = precision_recall_fscore_support(y_true_sample, preds_a_sample, average='binary', zero_division=0)
    _, _, f1_b_sample, _ = precision_recall_fscore_support(y_true_sample, preds_b_sample, average='binary', zero_division=0)

    bootstrap_diffs.append(f1_b_sample - f1_a_sample)

ci_lower = float(np.percentile(bootstrap_diffs, 2.5))
ci_upper = float(np.percentile(bootstrap_diffs, 97.5))
print(f"   - Rentang Bootstrap 95% CI: [{ci_lower:.4f} , {ci_upper:.4f}]")
print(f"     (Karena rentang tidak melewati angka 0, perbedaan performa dinyatakan signifikan secara statistik)")

# 4. Cohen's d (Effect Size)
mean_diff = np.mean(f1_scores_b) - np.mean(f1_scores_a)
pooled_std = np.sqrt((np.var(f1_scores_b, ddof=1) + np.var(f1_scores_a, ddof=1)) / 2)
cohens_d = float(mean_diff / pooled_std)
print(f"\n4. Cohen's d Effect Size:")
print(f"   - Nilai Cohen's d: {cohens_d:.2f}")
if cohens_d > 0.8:
    print("   - Tafsiran       : Large/Extremely Large Effect (Pengaruh Sangat Kuat)")

# 5. Analisis Kesalahan Linguistik Berdasarkan Kategori Teks
print("\n5. Menjalankan Subgroup Error Analysis (Analisis Kesalahan Linguistik)...")
test_sentences = test_dataset.texts
negation_words = {'not', 'no', 'never', "n't", 'neither', 'nor', 'without', 'lack', 'hardly', 'barely', 'scarcely'}
contrast_words = {'but', 'however', 'although', 'despite', 'yet', 'instead', 'whereas', 'though'}

cat_indices = {
    "Tanpa Negasi": [],
    "Negasi Biner": [],
    "Ironi / Sarkasme": [],
    "Review Panjang": [],
    "Ambiguitas Tinggi": []
}

for idx, text in enumerate(test_sentences):
    words = set(text.lower().split())
    token_len = len(text.split())
    has_negation = bool(words & negation_words)
    has_contrast = bool(words & contrast_words)

    if not has_negation:
        cat_indices["Tanpa Negasi"].append(idx)
    if has_negation:
        cat_indices["Negasi Biner"].append(idx)
    if has_contrast:
        cat_indices["Ironi / Sarkasme"].append(idx)
    if token_len > 20:
        cat_indices["Review Panjang"].append(idx)
    if has_negation and has_contrast:
        cat_indices["Ambiguitas Tinggi"].append(idx)

linguistic_error_results = []
print(f"\n   {'Kategori Linguistik':<22} | {'Sampel':<6} | {'Model A (Acc)':<13} | {'Model B (Acc)':<13}")
print("-" * 65)

for cat_name, indices in cat_indices.items():
    if len(indices) == 0:
        acc_a, acc_b = 0.0, 0.0
    else:
        y_sub = y_true_test[indices]
        pred_a_sub = preds_a_42[indices]
        pred_b_sub = preds_b_42[indices]

        acc_a = round(float(np.mean(pred_a_sub == y_sub) * 100), 1)
        acc_b = round(float(np.mean(pred_b_sub == y_sub) * 100), 1)

    linguistic_error_results.append({
        'category_name': cat_name,
        'model_a_accuracy': acc_a,
        'model_b_accuracy': acc_b,
        'sample_count': len(indices)
    })
    print(f"   {cat_name:<22} | {len(indices):<6} | {acc_a:>5.1f}%        | {acc_b:>5.1f}%")


=== MENJALANKAN UJI STATISTIK INFERENSIAL ===

1. Wilcoxon Test (n=6 F1-scores):
   - Statistik Uji : 21.0
   - p-value       : 0.01562 (Signifikan jika p < 0.05)

2. Tabel Kontingensi McNemar (Seed 42, N=872):
   - Kedua Model Benar (A+ / B+)                 : 717
   - Model B Benar & Model A Salah (A- / B+) (b) : 88
   - Model A Benar & Model B Salah (A+ / B-) (c) : 22
   - Kedua Model Salah (A- / B-)                 : 45
   - McNemar Chi2   : 38.4091
   - McNemar p-value: 0.00000000 (Signifikan jika p < 0.05)

3. Melakukan Bootstrap Resampling 10.000 kali...
   - Rentang Bootstrap 95% CI: [0.0501 , 0.0927]
     (Karena rentang tidak melewati angka 0, perbedaan performa dinyatakan signifikan secara statistik)

4. Cohen's d Effect Size:
   - Nilai Cohen's d: 12.72
   - Tafsiran       : Large/Extremely Large Effect (Pengaruh Sangat Kuat)

5. Menjalankan Subgroup Error Analysis (Analisis Kesalahan Linguistik)...

   Kategori Linguistik    | Sampel | Model A (Acc) | Model B (Acc)
-------

## **6. Eksport Hasil Eksperimen Langsung ke Database Web App (`app.db`)**
Bagian ini menuliskan secara langsung seluruh metrik run eksperimen dan hasil uji statistik ke SQLite database `app.db` agar langsung disajikan pada visualisasi dashboard web app.

In [8]:
db_path = "app.db"
print(f"Menyimpan data eksperimen ke database: {os.path.abspath(db_path)}")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

try:
    # 0. Buat tabel jika belum ada (CREATE TABLE IF NOT EXISTS)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS benchmark_results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            seed_number INTEGER NOT NULL,
            model_type VARCHAR(20) NOT NULL,
            accuracy FLOAT NOT NULL,
            precision FLOAT NOT NULL,
            recall FLOAT NOT NULL,
            f1_score FLOAT NOT NULL,
            inference_time_ms FLOAT NOT NULL,
            peak_vram_mb FLOAT NOT NULL
        );
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS statistical_tests (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            mcnemar_p_value FLOAT NOT NULL,
            wilcoxon_p_value FLOAT NOT NULL,
            bootstrap_ci_lower FLOAT NOT NULL,
            bootstrap_ci_upper FLOAT NOT NULL,
            cohens_d FLOAT NOT NULL,
            mcnemar_both_correct INTEGER DEFAULT 712,
            mcnemar_a_correct_b_wrong INTEGER DEFAULT 27,
            mcnemar_b_correct_a_wrong INTEGER DEFAULT 89,
            mcnemar_both_wrong INTEGER DEFAULT 44,
            mcnemar_chi2 FLOAT DEFAULT 32.0776,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS error_analysis_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name VARCHAR(50) NOT NULL,
            model_a_accuracy FLOAT NOT NULL,
            model_b_accuracy FLOAT NOT NULL,
            sample_count INTEGER NOT NULL,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS prediction_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username VARCHAR(50) DEFAULT 'public',
            input_text TEXT NOT NULL,
            model_a_label VARCHAR(20) NOT NULL,
            model_a_confidence FLOAT NOT NULL,
            model_a_latency_ms FLOAT NOT NULL,
            model_b_label VARCHAR(20) NOT NULL,
            model_b_confidence FLOAT NOT NULL,
            model_b_latency_ms FLOAT NOT NULL,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
    """)

    # 1. Bersihkan tabel hasil benchmark lama
    cursor.execute("DELETE FROM benchmark_results")

    # 2. Masukkan metrik baru untuk masing-masing seed
    for res in benchmark_results:
        cursor.execute("""
            INSERT INTO benchmark_results (seed_number, model_type, accuracy, precision, recall, f1_score, inference_time_ms, peak_vram_mb)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            res['seed'],
            res['model_type'],
            float(res['accuracy']),
            float(res['precision']),
            float(res['recall']),
            float(res['f1_score']),
            float(res['latency']),
            float(res['vram'])
        ))

    # 3. Bersihkan dan masukkan hasil uji statistik inferensial
    cursor.execute("DELETE FROM statistical_tests")
    cursor.execute("""
        INSERT INTO statistical_tests (
            mcnemar_p_value, wilcoxon_p_value, bootstrap_ci_lower, bootstrap_ci_upper, cohens_d,
            mcnemar_both_correct, mcnemar_a_correct_b_wrong, mcnemar_b_correct_a_wrong, mcnemar_both_wrong, mcnemar_chi2
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        float(mcnemar_p),
        float(wilcoxon_p),
        float(ci_lower),
        float(ci_upper),
        float(cohens_d),
        int(both_correct),
        int(a_correct_b_wrong),
        int(b_correct_a_wrong),
        int(both_wrong),
        float(mcnemar_chi2)
    ))

    # 4. Bersihkan dan masukkan hasil analisis kesalahan linguistik
    cursor.execute("DELETE FROM error_analysis_logs")
    for err in linguistic_error_results:
        cursor.execute("""
            INSERT INTO error_analysis_logs (category_name, model_a_accuracy, model_b_accuracy, sample_count)
            VALUES (?, ?, ?, ?)
        """, (
            err['category_name'],
            float(err['model_a_accuracy']),
            float(err['model_b_accuracy']),
            int(err['sample_count'])
        ))

    conn.commit()
    print("\n[SUKSES] Seluruh data eksperimen (Benchmark, Statistik, Error Analysis) berhasil diexport ke database SQLite (app.db)!")
except Exception as e:
    conn.rollback()
    print(f"\n[GAGAL] Terjadi kesalahan saat menyimpan ke database: {e}")
finally:
    conn.close()


Menyimpan data eksperimen ke database: /content/app.db

[SUKSES] Seluruh data eksperimen (Benchmark, Statistik, Error Analysis) berhasil diexport ke database SQLite (app.db)!


In [9]:
# 7. Kompres Bobot Model ke Berkas models.zip untuk Kemudahan Unduh dari Colab
import shutil
shutil.make_archive("models", 'zip', "models")
print("\n[SUKSES] Berkas 'models.zip' berhasil dibuat!")
print("Silakan unduh berkas 'app.db' dan 'models.zip' dari panel berkas kiri Google Colab Anda.")


[SUKSES] Berkas 'models.zip' berhasil dibuat!
Silakan unduh berkas 'app.db' dan 'models.zip' dari panel berkas kiri Google Colab Anda.
